# 组合策略回测 Demo

使用 `TrendFollowingStrategy`（ATR-RSI趋势跟踪）对茅台（600519.SSE）进行回测。

**前置条件**：已通过 DataManager 下载 600519.SSE 的日线数据。

## 第一步：初始化回测引擎

In [1]:
from datetime import datetime
from vnpy.trader.constant import Interval, Exchange
from vnpy_portfoliostrategy.backtesting import BacktestingEngine
from vnpy_portfoliostrategy.strategies.trend_following_strategy import TrendFollowingStrategy

# 创建回测引擎
engine = BacktestingEngine()

# 配置回测参数
engine.set_parameters(
    vt_symbols=["600519.SSE"],      # 回测标的
    interval=Interval.DAILY,        # K线周期：日线
    start=datetime(2020, 1, 1),     # 开始日期
    end=datetime(2024, 12, 31),     # 结束日期
    rates={"600519.SSE": 1/10000},  # 手续费率（万一）
    slippages={"600519.SSE": 0},    # 滑点
    sizes={"600519.SSE": 1},        # 合约乘数（A股为1）
    priceticks={"600519.SSE": 0.01},# 最小价格变动
    capital=1_000_000               # 初始资金 100 万
)

print("回测引擎配置完成")

回测引擎配置完成


## 第二步：加载策略

In [2]:
# 加载策略，可以修改参数
engine.add_strategy(
    strategy_class=TrendFollowingStrategy,
    setting={
        "atr_window": 22,
        "atr_ma_window": 10,
        "rsi_window": 5,
        "rsi_entry": 16,
        "trailing_percent": 0.8,
        "fixed_size": 100,   # 每次买 100 股（A股一手）
        "price_add": 0
    }
)

print("策略加载完成")

策略加载完成


## 第三步：加载历史数据

In [3]:
engine.load_data()
print("历史数据加载完成")

2026-06-02 18:47:25.723215	开始加载历史数据
2026-06-02 18:47:27.222192	600519.SSE历史数据加载完成，数据量：1212
2026-06-02 18:47:27.222325	所有历史数据加载完成
历史数据加载完成


## 第四步：运行回测

In [4]:
engine.run_backtesting()
print("回测运行完成")

2026-06-02 18:47:31.124486	策略初始化完成
2026-06-02 18:47:31.124878	开始回放历史数据
2026-06-02 18:47:31.143480	历史数据回放结束
回测运行完成


## 第五步：查看统计结果

In [5]:
df = engine.calculate_result()
stats = engine.calculate_statistics()
print("\n====== 回测统计结果 ======")
for k, v in stats.items():
    print(f"{k}: {v}")

2026-06-02 18:47:33.876084	开始计算逐日盯市盈亏
2026-06-02 18:47:33.892007	逐日盯市盈亏计算完成
2026-06-02 18:47:33.892933	开始计算策略统计指标
2026-06-02 18:47:33.929476	------------------------------
2026-06-02 18:47:33.929531	首个交易日：	2020-01-15
2026-06-02 18:47:33.929543	最后交易日：	2024-12-31
2026-06-02 18:47:33.929552	总交易日：	1203
2026-06-02 18:47:33.929559	盈利交易日：	157
2026-06-02 18:47:33.929564	亏损交易日：	161
2026-06-02 18:47:33.929572	起始资金：	1,000,000.00
2026-06-02 18:47:33.929580	结束资金：	985,496.41
2026-06-02 18:47:33.929586	总收益率：	-1.45%
2026-06-02 18:47:33.929591	年化收益：	-0.29%
2026-06-02 18:47:33.929596	最大回撤: 	-43,639.88
2026-06-02 18:47:33.929728	百分比最大回撤: -4.24%
2026-06-02 18:47:33.929747	最长回撤天数: 	144
2026-06-02 18:47:33.929754	总盈亏：	-14,503.59
2026-06-02 18:47:33.929760	总手续费：	4,342.59
2026-06-02 18:47:33.929766	总滑点：	0.00
2026-06-02 18:47:33.929771	总成交金额：	43,425,889.00
2026-06-02 18:47:33.929777	总成交笔数：	242
2026-06-02 18:47:33.929782	日均盈亏：	-12.06
2026-06-02 18:47:33.929787	日均手续费：	3.61
2026-06-02 18:47:33.929791	日均滑点：	0.00
2

## 第六步：绘制资金曲线

In [6]:
engine.show_chart()